# Federated Learning Experiments - Google Colab Setup

This notebook sets up the federated learning framework and runs experiments on the EBHI-SEG dataset.

**Features:**
- Auto-download dataset from figshare
- Compare FL algorithms: FedAvg, FedProx, FedOptimizer
- Compare models: UNet, DeepLabV3, FCN
- Save results to Google Drive

**GPU:** Uses Colab's free T4 GPU (auto-selected)

## Step 1: Mount Google Drive & Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive'

# Create project directory in Drive
PROJECT_DIR = os.path.join(DRIVE_PATH, 'federated_learning')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'data'), exist_ok=True)

print(f'✓ Google Drive mounted')
print(f'✓ Project directory: {PROJECT_DIR}')

## Step 2: Clone Repository

In [ ]:
import subprocess
import os
from getpass import getpass

# Clone the private repository to local Colab storage (fast I/O)
repo_path = '/content/federated_learning_repo'

if not os.path.exists(repo_path):
    print('Cloning private repository...\n')
    print('You will need a GitHub Personal Access Token (PAT) for private repo access.\n')
    print('To create a PAT:')
    print('  1. Go to https://github.com/settings/tokens')
    print('  2. Click "Generate new token (classic)"')
    print('  3. Enable "repo" scope')
    print('  4. Copy the token below\n')
    
    github_token = getpass('Enter your GitHub Personal Access Token: ')
    github_username = input('Enter your GitHub username: ')
    repo_name = input('Enter repository name (e.g., federated_learning): ')
    
    clone_url = f'https://{github_token}@github.com/{github_username}/{repo_name}.git'
    
    try:
        subprocess.run([
            'git', 'clone',
            clone_url,
            repo_path
        ], check=True, capture_output=True)
        print('\n✓ Repository cloned successfully')
    except subprocess.CalledProcessError as e:
        print(f'\n✗ Clone failed: {e.stderr.decode()}')
        print('\nAlternative: Upload repo as ZIP to Drive')
        print('  1. On your machine: zip -r federated_learning.zip federated_learning/')
        print('  2. Upload ZIP to MyDrive/federated_learning.zip')
        print('  3. Re-run this cell and choose the ZIP option')
        raise
else:
    print('✓ Repository already exists')

# Add to Python path
import sys
sys.path.insert(0, repo_path)
os.chdir(repo_path)

print(f'✓ Working directory: {repo_path}')


## Step 3: Install Dependencies

In [ ]:
import subprocess
import sys

print('Installing dependencies...')

# Core dependencies
packages = [
    'torch>=2.0',
    'torchvision',
    'pyyaml',
    'pillow',
    'numpy',
    'tqdm',
    'matplotlib',
    'scikit-learn',
    'albumentations',
    'requests',  # for downloading dataset
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)

print('✓ All dependencies installed')

## Step 4: Download Dataset

In [ ]:
import os
import shutil

# Paths
drive_dataset = os.path.join(DRIVE_PATH, 'EBHI-SEG')
data_dir = '/content/data/EBHI-SEG'

print('=' * 70)
print('DATASET SETUP')
print('=' * 70)

# Copy from Drive to local Colab storage
if not os.path.exists(data_dir):
    os.makedirs('/content/data', exist_ok=True)
    if os.path.exists(drive_dataset):
        print(f'Copying from: {drive_dataset}')
        shutil.copytree(drive_dataset, data_dir)
        print(f'✓ Dataset copied to: {data_dir}\n')
    else:
        raise FileNotFoundError(f"Dataset not found at {drive_dataset}")
else:
    print(f'Dataset already in local storage\n')

# Count and display images
print('Dataset structure:\n')
categories = sorted([d for d in os.listdir(data_dir) 
                    if os.path.isdir(os.path.join(data_dir, d)) 
                    and not d.startswith('.')])

total_images = 0
for category in categories:
    image_dir = os.path.join(data_dir, category, 'image')
    if os.path.exists(image_dir):
        count = len([f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))])
        total_images += count
        print(f'  {category:20} {count:>4} images')

print(f'\n{"="*70}')
print(f'Total: {total_images} images')
print(f'✓ Ready for training!')


In [ ]:
import yaml

# Update config to use the local data path
config_path = os.path.join(repo_path, 'configs', 'config.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['data_root'] = '/content/data/EBHI-SEG'

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✓ Updated config.yaml: data_root = {config["data_root"]}')


## Step 5: Verify Setup

In [ ]:
import torch
from src.model import create_model

print('Checking setup...')
print(f'✓ PyTorch version: {torch.__version__}')
print(f'✓ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

# Test models load
print(f'\n✓ Testing model factory...')
for model_type in ['unet', 'deeplab', 'fcn']:
    m = create_model(model_type, 3, 1)
    params = sum(p.numel() for p in m.parameters())
    print(f'  {model_type.upper():10} - {params:>10,} params')

print(f'\n✓ All systems ready!')

## Step 6: Run Baseline (Centralized Training)

In [ ]:
# Run centralized training baseline (UNet, 30 epochs)
# This establishes a baseline to compare federated learning against
print('Running centralized training baseline with UNet...')
print('This will train on the full dataset (no federation)\n')

!python main.py \
  --mode centralized \
  --model_type unet \
  --epochs 30 \
  --batch_size 16 \
  --device cuda \
  --seed 42

## Step 7: Experiment - Compare FL Algorithms

In [ ]:
# Compare FL algorithms: FedAvg vs FedProx vs FedOptimizer
# Using UNet, non-IID data (by_class), 20 FL rounds

algorithms = ['fedavg', 'fedprox', 'fedoptimizer']

for algo in algorithms:
    print(f'\n{"="*60}')
    print(f'Running: FedAvg simulation with algorithm={algo.upper()}')
    print(f'{"="*60}\n')
    
    !python main.py \
      --mode federated \
      --model_type unet \
      --fl_algorithm {algo} \
      --partition by_class \
      --fl_rounds 20 \
      --local_epochs 3 \
      --batch_size 16 \
      --device cuda \
      --seed 42

## Step 8: Experiment - Compare Models

In [ ]:
# Compare models: UNet vs DeepLabV3 vs FCN
# Using FedAvg, non-IID data (by_class), 20 FL rounds

models = ['unet', 'deeplab', 'fcn']

for model_type in models:
    print(f'\n{"="*60}')
    print(f'Running: Federated Learning with model={model_type.upper()}')
    print(f'{"="*60}\n')
    
    !python main.py \
      --mode federated \
      --model_type {model_type} \
      --fl_algorithm fedavg \
      --partition by_class \
      --fl_rounds 20 \
      --local_epochs 3 \
      --batch_size 16 \
      --device cuda \
      --seed 42

## Step 9: Compare Non-IID vs IID Data Distribution

In [ ]:
# Experiment: Effect of data distribution (Non-IID vs IID)
# Using UNet + FedAvg

distributions = [
    ('by_class', 'Non-IID (each hospital specializes in one class)'),
    ('random', 'IID (data randomly distributed across hospitals)'),
]

for strategy, description in distributions:
    print(f'\n{"="*60}')
    print(f'{description}')
    print(f'{"="*60}\n')
    
    !python main.py \
      --mode federated \
      --model_type unet \
      --fl_algorithm fedavg \
      --partition {strategy} \
      --n_clients 6 \
      --fl_rounds 20 \
      --local_epochs 3 \
      --device cuda \
      --seed 42

## Step 10: Load and Visualize Results

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

# Load results from all experiments
outputs_dir = Path('outputs')

print('Experiment Results Summary:')
print('='*70)

# Centralized results
centralized_dir = outputs_dir / 'centralized'
if (centralized_dir / 'fl_history.json').exists():
    with open(centralized_dir / 'fl_history.json') as f:
        cent_history = json.load(f)
    print(f'\n✓ Centralized Training (UNet)')
    
# Federated results
federated_dir = outputs_dir / 'federated'
if (federated_dir / 'fl_history.json').exists():
    with open(federated_dir / 'fl_history.json') as f:
        fed_history = json.load(f)
    if 'rounds' in fed_history:
        latest_round = fed_history['rounds'][-1]
        print(f'\n✓ Federated Training (Latest Round)')
        print(f'  Dice:  {latest_round.get("dice", "N/A"):.4f}')
        print(f'  IoU:   {latest_round.get("iou", "N/A"):.4f}')
        print(f'  Loss:  {latest_round.get("loss", "N/A"):.4f}')

print('\n' + '='*70)
print('Results saved to: outputs/')
print('Synced to Google Drive:')
print(f'  {os.path.join(DRIVE_PATH, "federated_learning_repo", "outputs")}')

## Step 11: Copy Results to Google Drive

In [ ]:
import shutil
from pathlib import Path

# Copy results to Drive for safekeeping
src = Path('outputs')
dst = Path(DRIVE_PATH) / 'federated_learning_results'

if src.exists():
    print(f'Copying results to Google Drive...')
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'✓ Results saved to: {dst}')
else:
    print('No results directory found')

## Summary

You've successfully run:

1. **Baseline:** Centralized training (UNet)
2. **Algorithm Comparison:** FedAvg vs FedProx vs FedOptimizer
3. **Model Comparison:** UNet vs DeepLabV3 vs FCN
4. **Data Distribution:** Non-IID vs IID effects

### Next Steps:
- Download results from Google Drive
- Analyze metrics (Dice, IoU, convergence speed)
- Plot comparison figures
- Write up findings

### Troubleshooting:
- **Out of memory:** Reduce `batch_size` or `fl_rounds`
- **Session timeout:** Experiments auto-save checkpoints
- **Network issues:** Dataset downloads resume automatically